In [13]:
from pathlib import Path
import sys
parent_dir = Path.cwd().parent
sys.path.insert(0, str(parent_dir))
from config import BIGQUERY_API_KEY
from google.cloud import bigquery
from google.api_core.exceptions import GoogleAPIError

import hashlib
import json
import time
from datetime import datetime, timedelta
import pandas as pd
import requests
from bs4 import BeautifulSoup

client = bigquery.Client(project="proven-reality-499800-u9")



/Users/lloydtodaro/anaconda3/envs/nascar-visibility/lib/python3.11/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [23]:
# initialize everything

from pathlib import Path

SPONSORS = ["Progressive", "Cheddar's Scratch Kitchen", "Love's Travel Stops", "Busch Light", "Castrol"]

SEASONS = [
    ("2025-02-01", "2025-12-01"),
    ("2026-02-01", "2026-12-01"),
]

DOC_API_ROLLING_DAYS = 85

SPORTS_DOMAINS = [
    "nascar.com", "jayski.com", "frontstretch.com", "foxsports.com",
    "espn.com", "si.com", "motorsport.com", "autoweek.com", "racer.com",
    "nbcsports.com", "usatoday.com", "cbssports.com", "bleacherreport.com",
    "athlonsports.com", "sportsnaut.com", "motorsportweek.com",
    "speedwaydigest.com",
]

URL_KEYWORDS = ["nascar", "cup-series", "xfinity-series", "truck-series", "daytona", "talladega"]

OUTPUT_CSV = Path("data/raw/sponsor_mentions.csv")
CHECKPOINT_FILE = Path("sponsor_mentions_checkpoint.json")
FAILED_LOG = Path("sponsor_mentions_failed.json")
CANDIDATE_CACHE_DIR = Path("candidate_cache")   # one JSON per month of BQ candidate URLs
ARTICLE_TEXT_CACHE_DIR = Path("article_text_cache")  # one .txt per scraped URL

DOC_API_URL = "https://api.gdeltproject.org/api/v2/doc/doc"
REQUEST_HEADERS = {"User-Agent": "Mozilla/5.0 (research script; contact: lltodaro@gmail.com)"}

CANDIDATE_CACHE_DIR.mkdir(exist_ok=True)
ARTICLE_TEXT_CACHE_DIR.mkdir(exist_ok=True)

MAX_GB_WARNING = 5.0  # print a warning if a single query scans more than this

QUERY = """
    SELECT
      DATE,
      DocumentIdentifier AS url,
      SourceCommonName AS source,
      V2Organizations,
      V2Tone
    FROM `gdelt-bq.gdeltv2.gkg_partitioned`
    WHERE _PARTITIONTIME >= TIMESTAMP(@start_date)
      AND _PARTITIONTIME <  TIMESTAMP(@end_date)
      AND V2Organizations LIKE @sponsor_pattern
      AND (LOWER(V2Organizations) LIKE '%nascar%' OR LOWER(AllNames) LIKE '%nascar%')
"""


In [15]:
def month_windows(start_str, end_str):
    start = pd.Timestamp(start_str)
    end = pd.Timestamp(end_str)
    windows = []
    cur = start
    while cur < end:
        nxt = min(cur + pd.offsets.MonthBegin(1), end)
        windows.append((cur.strftime("%Y-%m-%d"), nxt.strftime("%Y-%m-%d")))
        cur = nxt
    return windows

In [16]:
import json
def load_checkpoint():
    """Load the set of (sponsor, start, end) jobs that already completed successfully."""
    if CHECKPOINT_FILE.exists():
        return set(tuple(x) for x in json.loads(CHECKPOINT_FILE.read_text()))
    return set()

 
def save_checkpoint(done):
    CHECKPOINT_FILE.write_text(json.dumps([list(x) for x in done]))
    
def log_failure(sponsor, start, end, error):
    failures = []
    if FAILED_LOG.exists():
        failures = json.loads(FAILED_LOG.read_text())
    failures.append({"sponsor": sponsor, "start": start, "end": end, "error": str(error)})
    FAILED_LOG.write_text(json.dumps(failures, indent=2))
 
 
  
def append_results(rows):
    if not rows:
        return
    df = pd.DataFrame(rows)
    write_header = not OUTPUT_CSV.exists()
    df.to_csv(OUTPUT_CSV, mode="a", header=write_header, index=False)
    
def url_hash(url):
    return hashlib.sha256(url.encode("utf-8")).hexdigest()[:24]
 
 

In [18]:
# recent dates:

def query_doc_api(sponsor, start, end):
    """Full-text search via GDELT DOC 2.0 API. Returns a list of article dicts."""
    start_dt = start.replace("-", "") + "000000"
    end_dt = end.replace("-", "") + "235959"
 
    params = {
        "query": f'"{sponsor}" NASCAR',
        "mode": "artlist",
        "format": "json",
        "maxrecords": 250,
        "sort": "datedesc",
        "STARTDATETIME": start_dt,
        "ENDDATETIME": end_dt,
    }
 
    resp = requests.get(DOC_API_URL, params=params, headers=REQUEST_HEADERS, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    articles = data.get("articles", [])
 
    if len(articles) == 250:
        print(f"    note: hit the 250-record cap for {sponsor} {start} to {end} - "
              f"consider splitting this window smaller if completeness matters")
 
    rows = []
    for a in articles:
        rows.append({
            "sponsor": sponsor,
            "url": a.get("url"),
            "title": a.get("title"),
            "domain": a.get("domain"),
            "seendate": a.get("seendate"),
            "source_method": "doc_api",
        })
    return rows
 

In [ ]:
# old dates via bigquery:

def get_candidate_urls(client, start, end):
    """Get (and cache) candidate article URLs from known sports domains for one month."""
    cache_file = CANDIDATE_CACHE_DIR / f"{start}_{end}.json"
    if cache_file.exists():
        return json.loads(cache_file.read_text())
 
    query = """
        SELECT DISTINCT DocumentIdentifier AS url, SourceCommonName AS domain, DATE
        FROM `gdelt-bq.gdeltv2.gkg_partitioned`
        WHERE _PARTITIONTIME >= TIMESTAMP(@start_date)
          AND _PARTITIONTIME <  TIMESTAMP(@end_date)
          AND SourceCommonName IN UNNEST(@domains)
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("start_date", "STRING", start),
            bigquery.ScalarQueryParameter("end_date", "STRING", end),
            bigquery.ArrayQueryParameter("domains", "STRING", SPORTS_DOMAINS),
        ]
    )
 
    dry_config = bigquery.QueryJobConfig(
        dry_run=True, use_query_cache=False, query_parameters=job_config.query_parameters,
    )
    dry_job = client.query(query, job_config=dry_config)
    gb_scanned = dry_job.total_bytes_processed / 1e9
    if gb_scanned > MAX_GB_WARNING:
        print(f"    warning: candidate query for {start} to {end} will scan {gb_scanned:.2f} GB")
 
    df = client.query(query, job_config=job_config).to_dataframe()
    candidates = df.to_dict("records")
    candidates = [c for c in candidates if is_likely_nascar_url(c["url"])]
    cache_file.write_text(json.dumps(candidates, default=str))
    return candidates


def fetch_article_text(url):
    """Fetch and cache the visible text of a URL. Returns None on failure."""
    cache_file = ARTICLE_TEXT_CACHE_DIR / f"{url_hash(url)}.txt"
    if cache_file.exists():
        return cache_file.read_text(encoding="utf-8", errors="ignore")

    try:
        resp = requests.get(url, headers=REQUEST_HEADERS, timeout=15)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        for tag in soup(["script", "style", "nav", "header", "footer"]):
            tag.decompose()
        text = soup.get_text(separator=" ", strip=True)
    except Exception:
        text = ""  # cache the failure too, as empty, so we don't retry every run

    cache_file.write_text(text, encoding="utf-8")
    return text


def scan_candidates_for_sponsor(sponsor, candidates):
    """Search cached/fetched article text for a sponsor mention."""
    rows = []
    sponsor_lower = sponsor.lower()
    for c in candidates:
        
        text = fetch_article_text(c["url"])
        if text and sponsor_lower in text.lower():
            rows.append({
                "sponsor": sponsor,
                "url": c["url"],
                "title": None,
                "domain": c.get("domain"),
                "seendate": c.get("DATE"),
                "source_method": "bq_candidate_scrape",
            })
        time.sleep(0.2)  # be polite between page fetches
    return rows

def is_likely_nascar_url(url):
    url_lower = url.lower()
    return any(keyword in url_lower for keyword in URL_KEYWORDS)

In [26]:
client = bigquery.Client(project="proven-reality-499800-u9")
done = load_checkpoint()

cutoff = datetime.utcnow() - timedelta(days=DOC_API_ROLLING_DAYS)

all_windows = []
for season_start, season_end in SEASONS:
    all_windows.extend(month_windows(season_start, season_end))

# Pre-fetch candidate URLs once per historical month (shared across all sponsors)
historical_windows = [w for w in all_windows if pd.Timestamp(w[1]) <= cutoff]
candidates_by_window = {}
for start, end in historical_windows:
    print(f"fetching BigQuery candidates for {start} to {end}")
    try:
        candidates_by_window[(start, end)] = get_candidate_urls(client, start, end)
    except GoogleAPIError as e:
        print(f"    failed: {e}")
        log_failure("ALL", start, end, "bq_candidates", e)
        candidates_by_window[(start, end)] = []

total_jobs = len(SPONSORS) * len(all_windows)
job_num = 0

for sponsor in SPONSORS:
    for start, end in all_windows:
        job_num += 1
        is_recent = pd.Timestamp(end) > cutoff
        method = "doc_api" if is_recent else "bq_scrape"
        key = (sponsor, start, end, method)

        if key in done:
            print(f"[{job_num}/{total_jobs}] skip  {sponsor} {start} to {end} ({method}, already done)")
            continue

        print(f"[{job_num}/{total_jobs}] {method:10s} {sponsor} {start} to {end}")
        try:
            if is_recent:
                rows = query_doc_api(sponsor, start, end)
            else:
                candidates = candidates_by_window.get((start, end), [])
                rows = scan_candidates_for_sponsor(sponsor, candidates)
        except Exception as e:
            print(f"    failed: {e}")
            log_failure(sponsor, start, end, method, e)
            continue

        append_results(rows)
        print(f"    found {len(rows)} mentions")

        done.add(key)
        save_checkpoint(done)
        time.sleep(0.3)

print("\nDone. Results in:", OUTPUT_CSV.resolve())
if FAILED_LOG.exists():
    print("Some queries failed - see:", FAILED_LOG.resolve())


/Users/lloydtodaro/anaconda3/envs/nascar-visibility/lib/python3.11/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


fetching BigQuery candidates for 2025-02-01 to 2025-03-01


/Users/lloydtodaro/anaconda3/envs/nascar-visibility/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


FileNotFoundError: [Errno 2] No such file or directory: 'candidate_cache/2025-02-01_2025-03-01.json'

In [24]:
import sys
print(sys.executable)

/Users/lloydtodaro/anaconda3/envs/nascar-visibility/bin/python


In [25]:
%pip install db-dtypes

Note: you may need to restart the kernel to use updated packages.


In [11]:
sql = """
SELECT COUNT(*) as cnt
FROM `gdelt-bq.gdeltv2.gkg_partitioned`
WHERE _PARTITIONTIME >= TIMESTAMP("2025-06-01")
  AND _PARTITIONTIME <  TIMESTAMP("2025-07-01")
  AND LOWER(V2Organizations) LIKE '%nascar%'
"""

df = client.query(sql).to_dataframe()
print(df)

/Users/lloydtodaro/anaconda3/envs/nascar-visibility/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


   cnt
0    1


In [12]:
sql = """
SELECT COUNT(*) as cnt
FROM `gdelt-bq.gdeltv2.gkg_partitioned`
WHERE _PARTITIONTIME >= TIMESTAMP("2025-06-01")
  AND _PARTITIONTIME <  TIMESTAMP("2025-07-01")
  AND LOWER(V2Organizations) LIKE '%google%'
"""
df = client.query(sql).to_dataframe()
print(df)

/Users/lloydtodaro/anaconda3/envs/nascar-visibility/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


     cnt
0  27406
